# Dawuro - architecture companion

**Scripture as conversation on WhatsApp.** Dawuro (Akan: the town crier's gong) is a mobile web app built for the *Scripture in New Frontiers* hackathon (YouVersion + Gloo AI). A person speaks or types a feeling in English or an African language; Gloo AI chooses the best-fitting verse from a curated allow-list; YouVersion supplies the actual Scripture words in English and the local language; the verse is spoken aloud; Gloo writes a short tradition-aware reflection; and the share that goes out on WhatsApp carries a link that lets the receiver - no app, no login - hear the verse and reply with one of their own into the same chat. A daily notification can bring that day's verse in the user's language. This notebook is the code-verification companion for judges: it walks the architecture and shows the exact shape of every API call the app makes.

- Live app: https://dawuro.vercel.app
- Repository: https://github.com/NasamuAlhassan/dawuro

The illustrative code cells below mirror the app's server-side calls. **They require your own keys** (YouVersion, Gloo, GhanaNLP Khaya) supplied via Kaggle secrets - no keys ship with this notebook. Everything they demonstrate can also be verified by reading the repo files referenced in each section.

## The loop

```
Sender (phone)
  type feeling  OR  mic (EN: Web Speech / local: MediaRecorder)
       |                      |
       |                      v
       |              POST /api/transcribe -> Khaya ASR
       |                      |
       v                      v
              POST /api/verse
                |  Gloo picks from curated allow-list (primary)
                |  keyword map fallback . YouVersion EN + local text
                v
              POST /api/reflect -> Gloo Completions v2 (tradition-aware)
                v
              POST /api/speak   -> voice provider ladder (see below)
                v
              ShareSheet -> PNG + audio + receive link
                |
                v  WhatsApp message
Receiver (any phone, no app)
  link unfurls as a verse card -- /v/{lang}/{usfm}/opengraph-image
       |
       v
  /v/{lang}/{usfm} -- server-rendered verse EN + local, Play button,
  publisher attribution, then "Your turn" -> same feeling->verse flow
       |
       v
  replies with a verse of their own -> the loop continues
```

## Architecture walkthrough

Next.js (App Router) + TypeScript, deployed on Vercel. Every vendor call happens in server route handlers - keys never reach the client. The app has three tabs: **Home** (the day's verse, big and playable), **Heart** (a voice-first "what's on your heart?" with a large microphone and feeling tiles), and **Settings** (language, daily reminder, reflection tone).

| Layer | File(s) | What it does |
|---|---|---|
| Scripture | `lib/youversion.ts` | Passages, Verse of the Day, bilingual assembly, caching, license errors |
| Verse selection | `lib/verses.ts`, `lib/gloo.ts`, `app/api/verse/route.ts` | Curated feeling->USFM map; Gloo picks from that allow-list as the primary brain; one-tap chips stay curated for a deterministic demo |
| Reflection | `lib/gloo.ts`, `app/api/reflect/route.ts` | Completions v2, `tradition` parameter, optional Khaya translation of the reflection |
| Voice | `lib/tts.ts`, `lib/abena.ts`, `lib/edge-tts.ts`, `lib/khaya.ts`, `lib/hf-tts.ts`, `lib/pregen-audio.ts` | A provider switcher with a fallback ladder, plus pre-generated offline audio (see "The voice stack") |
| Notifications | `lib/push.ts`, `app/api/push/*`, `public/sw.js`, `vercel.json` | Daily verse as a web push, in the user's language |
| Share | `components/ShareSheet.tsx`, `lib/card.ts`, `lib/share.ts` | PNG card (html-to-image, 1080x1350), audio file, receive link in the message text and PNG footer |
| Receive | `app/v/[lang]/[usfm]/page.tsx`, `opengraph-image.tsx`, `components/ReceiveClient.tsx`, `components/VerseFlow.tsx` | Server-rendered verse page + dynamic OG verse card + the reply flow |

Twenty languages are configured in `lib/languages.ts`: an English-first mode (BSB, read by the browser or Edge neural voice) plus nineteen African languages. When a language has a published Bible on YouVersion (Asante Twi ASNA #2094, Ewe ECS #1613, Yoruba YCB #911, ...), that text is used. When it does not (Kusaal, Ga, Dagbani, Fante, Dholuo, Kimeru), the local side is a clearly labelled Khaya translation of the YouVersion English verse - and English (BSB) remains the published Scripture on the card.

## YouVersion Platform API - the only source of Scripture words

`lib/youversion.ts`. Auth is a single header, `X-YVP-App-Key`.

Endpoints used:

- `GET /v1/bibles/{bible_id}/passages/{usfm}?format=text` - passage text (e.g. bible 3034 = English BSB, 2094 = Asante Twi ASNA)
- `GET /v1/bibles/{bible_id}` - version metadata, used to resolve the copyright line shown on every card
- `GET /v1/verse_of_the_days/{day}` - day-of-year (1-366) -> a USFM passage id for the Home verse
- `GET /v1/verse_of_the_days` - the full year list, used to pre-generate daily-verse audio

Bilingual assembly (`getBilingualPassage`): fetch English BSB first, then either the local published Bible (Path A) or a Khaya translation of that English text (Path B, labelled `source: "khaya"` with a `proxyNote`). Results are cached in-memory for 30 minutes per `{language}:{usfm}` key, with an in-flight promise map so the page, its metadata, and its OG image share one fetch. Copyright lines are cached separately.

**403 handling:** Twi Bibles require accepting the Biblica Fast-track Bible License in the YouVersion developer portal. The client maps a 403 to a `LICENSE_REQUIRED` error whose message tells the operator exactly where to click. `GET /api/health` probes `JHN.3.16` in Twi and reports `twiAccess.ok`.

In [ ]:
# YouVersion passage fetch -- mirrors lib/youversion.ts (requires your own keys)
import requests

# Kaggle: Add-ons -> Secrets -> attach YVP_APP_KEY to this notebook
from kaggle_secrets import UserSecretsClient
YVP_APP_KEY = UserSecretsClient().get_secret("YVP_APP_KEY")

YVP_BASE = "https://api.youversion.com/v1"
HEADERS = {"X-YVP-App-Key": YVP_APP_KEY}

BSB_ENGLISH = 3034      # Berean Standard Bible
ASANTE_TWI = 2094       # Asante Twi Nkwa Asem (needs Biblica Fast-track license)
USFM = "PHP.4.6-7"      # the demo verse

for bible_id, label in [(BSB_ENGLISH, "English BSB"), (ASANTE_TWI, "Asante Twi ASNA")]:
    r = requests.get(f"{YVP_BASE}/bibles/{bible_id}/passages/{USFM}", params={"format": "text"}, headers=HEADERS)
    if r.status_code == 403:
        print(f"{label}: 403 -- accept the Biblica Fast-track license at platform.youversion.com")
        continue
    r.raise_for_status()
    data = r.json()
    print(f"{label}: {data['reference']}\n{data['content'][:160]}...\n")

votd = requests.get(f"{YVP_BASE}/verse_of_the_days/206", headers=HEADERS).json()
print("Verse of the Day (day 206):", votd)

Response shapes the app relies on:

```json
// GET /v1/bibles/2094/passages/PHP.4.6-7?format=text
{
  "id": "PHP.4.6-7",
  "reference": "Philippians 4:6-7",
  "content": "Mommma biribiara nha mo. Daa, mo nkotosre ne aseda ..."
}

// GET /v1/verse_of_the_days/206
{ "day": 206, "passage_id": "PSA.46.1" }

// GET /v1/bibles/2094  (fields used)
{ "id": 2094, "abbreviation": "ASNA", "title": "Asante Twi Nkwa Asem", "copyright": "Biblica ..." }
```

## Gloo AI - the brain that chooses (and never writes) Scripture

`lib/gloo.ts`. Two jobs:

**1. Verse mapping (primary).** `POST /api/verse` always computes the curated keyword match first as a safety net, then asks Gloo to pick the best verse for the free-text feeling - but only from the curated USFM allow-list built from `lib/verses.ts`. The prompt demands a JSON-only reply (`{"usfm": "PHP.4.6-7", "topic": "Anxiety"}`), and the server validates `allowedReferences.includes(usfm)` before using it. Any failure - no keys, HTTP error, 8-second timeout (AbortController, covering the token fetch too), unparseable output, or a reference outside the list - returns `null` and the curated pick stands. One-tap suggestion chips and the demo phrase skip Gloo entirely so the filmed path is reproducible. Gloo can choose Scripture; it cannot write it.

**2. Reflection.** Completions v2 with the `tradition` parameter (`evangelical` | `catholic` | `mainline`) and `auto_routing: true` writes a 2-3 sentence pastoral reflection. The system prompt forbids quoting or rewriting the verse and inventing Scripture. The reflection can then be Khaya-translated into the local language.

**Auth:** OAuth2 client-credentials against `https://platform.ai.gloo.com/oauth2/token` - HTTP Basic with `client_id:client_secret`, body `grant_type=client_credentials&scope=api/access`. The bearer token is cached in-process until 60 seconds before expiry.

**Degradation:** with no Gloo keys, `/api/reflect` returns a 503 with code `GLOO_NOT_CONFIGURED` (the UI shows a soft note) and verse mapping falls back to the curated map. The loop never breaks.

In [ ]:
# Gloo OAuth + allow-list verse mapping -- mirrors lib/gloo.ts (requires your own keys)
import base64, json, requests

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GLOO_CLIENT_ID = secrets.get_secret("GLOO_CLIENT_ID")
GLOO_CLIENT_SECRET = secrets.get_secret("GLOO_CLIENT_SECRET")

# 1) OAuth2 client-credentials token
basic = base64.b64encode(f"{GLOO_CLIENT_ID}:{GLOO_CLIENT_SECRET}".encode()).decode()
tok = requests.post(
    "https://platform.ai.gloo.com/oauth2/token",
    headers={"Authorization": f"Basic {basic}", "Content-Type": "application/x-www-form-urlencoded"},
    data={"grant_type": "client_credentials", "scope": "api/access"},
)
tok.raise_for_status()
token = tok.json()["access_token"]

# 2) Feeling -> verse, restricted to the curated allow-list (subset shown)
ALLOWED = ["PHP.4.6-7", "MAT.6.34", "1PE.5.7", "ISA.41.10", "PSA.34.18",
           "ROM.15.13", "ISA.40.31", "JHN.14.27", "PRO.3.5-6", "JHN.3.16"]
feeling = "I can't sleep before results day"

resp = requests.post(
    "https://platform.ai.gloo.com/ai/v2/chat/completions",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={
        "messages": [
            {"role": "system", "content":
             "You map a person's feeling to ONE Bible verse reference from an allow-list. "
             'Reply with ONLY compact JSON: {"usfm":"PHP.4.6-7","topic":"Anxiety"}. '
             "usfm MUST be exactly one of the allowed values. Do not invent verses or write Scripture text."},
            {"role": "user", "content": f'Feeling: "{feeling}"\nAllowed usfm: {", ".join(ALLOWED)}\nPick the best match.'},
        ],
        "auto_routing": True,
        "temperature": 0.2,
        "max_tokens": 80,
    },
    timeout=8,  # the app aborts at 8s and falls back to the curated keyword map
)
resp.raise_for_status()
raw = resp.json()["choices"][0]["message"]["content"]
pick = json.loads(raw)
assert pick["usfm"] in ALLOWED, "server-side allow-list validation -- reject and fall back"
print("Gloo chose:", pick)  # YouVersion, not Gloo, supplies the verse text

Completions v2 response shape (fields the app reads):

```json
{
  "choices": [ { "message": { "role": "assistant", "content": "{\"usfm\":\"PHP.4.6-7\",\"topic\":\"Anxiety\"}" } } ],
  "model": "...",
  "tradition": "evangelical"
}
```

The reflection call is the same endpoint with a pastoral system prompt, `tradition`, `temperature: 0.7`, `max_tokens: 220` - and its `content` is prose, never verse text.

## GhanaNLP Khaya - the hosted voice

`lib/khaya.ts`. Auth header `Ocp-Apim-Subscription-Key`, 25-second default timeout on every call. Three capabilities:

- **TTS** - `POST /tts/v1/synthesize` with `{"text": ..., "language": "tw"}` returns WAV. Verified live for `tw`, `ee`, `ki`; Fante text uses the Twi voice. Cached in-memory by `sha256(lang + text)`.
- **Translate** - `POST /v1/translate` with `{"in": text, "lang": "en-tw"}`. Khaya accepts ~1000 characters per call, so longer passages are chunked at sentence boundaries and rejoined. Used for local card text when no YouVersion Bible exists, feelings -> English for mapping, and reflections -> local. **Never used for Scripture when a published Bible exists.**
- **ASR** - `POST /asr/v1/transcribe?language=tw` with raw audio. Wired for Twi, Ewe, Ga, Dagbani, Kusaal, Yoruba. English speech uses the browser's Web Speech API.

**Base URL, and a real-world fix:** the public host `translation-api.ghananlp.org` sits behind Cloudflare bot protection that 403-challenges datacenter IPs (every call from Vercel returned a "Just a moment..." page during development). The app calls the Azure API Management origin `https://translation-ghananlp.azure-api.net` instead - the same API and key with no challenge - overridable via `KHAYA_API_BASE`. Raw upstream error bodies are logged server-side only; users always get a calm sentence and can still read the verse.

In [ ]:
# Khaya TTS + translate -- mirrors lib/khaya.ts (requires your own keys)
import requests

from kaggle_secrets import UserSecretsClient
KHAYA_API_KEY = UserSecretsClient().get_secret("KHAYA_API_KEY")

# The APIM origin, to avoid the Cloudflare challenge on the public host
BASE = "https://translation-ghananlp.azure-api.net"
HEADERS = {"Ocp-Apim-Subscription-Key": KHAYA_API_KEY, "Content-Type": "application/json"}

# TTS: Twi verse text (from YouVersion) -> WAV bytes
tts = requests.post(
    f"{BASE}/tts/v1/synthesize",
    headers=HEADERS,
    json={"text": "Mommma biribiara nha mo", "language": "tw"},
    timeout=25,
)
tts.raise_for_status()
print(f"TTS: {len(tts.content)} bytes, content-type {tts.headers.get('content-type')}")

# Translate: English -> Kusaal (only because Kusaal has no YouVersion Bible)
tr = requests.post(
    f"{BASE}/v1/translate",
    headers=HEADERS,
    json={"in": "Do not be anxious about anything.", "lang": "en-kus"},
    timeout=25,
)
tr.raise_for_status()
print("Translate en-kus:", tr.json())  # plain string or {"translatedText": ...}

## The voice stack - a fallback ladder, so the voice never dies

Voice is the heart of the app, so it does not depend on a single vendor. `lib/tts.ts` is a provider switcher; `POST /api/speak` walks a ladder and returns the first engine that answers, tagging the response with `X-Dawuro-Audio-Source`.

| Engine | File | Covers | Key needed |
|---|---|---|---|
| Pre-generated offline audio | `lib/pregen-audio.ts` | Twi, for every curated + daily verse | none (static files) |
| Abena AI | `lib/abena.ts` | Twi + Ghanaian-accented English | `ABENA_API_KEY` |
| Microsoft Edge neural | `lib/edge-tts.ts` | English (West African `en-NG` voice) | none |
| GhanaNLP Khaya | `lib/khaya.ts` | Twi, Ewe, Gikuyu | `KHAYA_API_KEY` |
| Meta MMS via Hugging Face | `lib/hf-tts.ts` | Twi/Akan, Ewe, Gikuyu, Hausa, Yoruba, Swahili | `HF_TOKEN` |

**The offline tier is the one that matters most.** Because Dawuro's verses come from fixed sets (the curated topical map plus the 365-day Verse of the Day), all 375 were synthesized once in Twi with Meta's open **MMS-tts-aka** model and shipped as static MP3 under `public/audio/tw/`, listed in a manifest. `getBilingualPassage` attaches that static `audioUrl` to covered Twi verses, so the play button works with **no API, no key, no quota** - it survived a multi-day outage of the hosted Khaya TTS during this build. When Khaya recovers, its higher-quality voice takes over automatically. English is spoken by Edge's free neural voice (a real, shareable MP3, not just the browser voice). This is the same open-model philosophy the DONDO section describes, already in production for TTS.

## The receive link and the OG verse card

This is what turns a share into a conversation. Files: `lib/share.ts`, `app/v/[lang]/[usfm]/page.tsx`, `app/v/[lang]/[usfm]/opengraph-image.tsx`, `components/ReceiveClient.tsx`, `components/VerseFlow.tsx`.

- Every share (message text and the PNG card footer) carries `NEXT_PUBLIC_APP_URL` + `/v/{lang}/{usfm}` - e.g. `https://dawuro.vercel.app/v/tw/PHP.4.6-7`.
- **Link preview:** the route's `opengraph-image.tsx` renders a 1200x630 PNG on the fly with Next.js `ImageResponse` - the deep fired-clay card with gong-gold arcs, reference, local text (a Google-Fonts glyph subset keeps the Ghanaian-orthography letters crisp), English, publisher attribution, and "Tap to hear it aloud". It is CDN-cached and never calls Khaya, so a crawler cannot trigger paid work. WhatsApp shows this card in the chat before anyone taps.
- **Receive page:** server-rendered - the verse is in the HTML, in the receiver's language and English, with a Play button and attribution. The USFM and language are validated; a bad link renders a `notFound()` page, not a crash.
- **The reply:** below the verse, "Your turn - What's on your heart?" mounts the same `VerseFlow` used on the Heart page. The receiver speaks or types, gets their own verse, and the share button reads "Send your verse back". No account was created at any point.

`NEXT_PUBLIC_APP_URL` is required in production (with a Vercel-URL fallback): receive links and OG previews are built from it.

## Daily verse notifications - the right word at the right moment

`lib/push.ts`, `app/api/push/{subscribe,send,daily}/route.ts`, `public/sw.js`, `vercel.json`. This lands one of the hackathon's named frontiers (a timely word on the device) with real Web Push, not a mock.

- **Subscribe:** Settings has a "Turn on daily verse" toggle. It registers the service worker, requests permission, subscribes with the VAPID public key, and POSTs the subscription and chosen language to `/api/push/subscribe`.
- **The notification:** VAPID-authenticated (`web-push`). Title is the reference, body is the verse in the subscriber's language, and tapping it opens `/v/{lang}/{usfm}` - the receive page. The service worker handles the `push` and `notificationclick` events.
- **Daily delivery:** a Vercel cron (`vercel.json`, 06:00) hits `/api/push/daily`, which builds each language's payload once and fans out to every subscriber, pruning endpoints that return 404/410. It is guarded by `CRON_SECRET`.
- **Storage:** subscriptions persist to Upstash Redis (REST) when configured, in-memory otherwise. "Send me one now" needs no store - the client hands the server its own subscription - so the feature is demoable instantly.

## Theology guardrails

1. **Published Scripture is never machine-translated.** If YouVersion has a Bible in the language, that text - and only that text - is shown.
2. **AI never writes Scripture.** Gloo selects a USFM reference from a curated allow-list; the server rejects anything outside the list; YouVersion supplies the words.
3. **Machine translation is labelled.** For languages with no YouVersion Bible, the local text carries `source: "khaya"`, a visible "not a published Bible" note, and English (BSB) stays on the card and in the share message as the published Scripture.
4. **Attribution everywhere.** Publisher copyright (Biblica, Berean Standard Bible, ...) is resolved from YouVersion metadata and printed on the verse card, the shared PNG, the OG link preview, and the share message text.
5. **Reflections are commentary, not canon.** The prompt forbids quoting or rewriting the verse; traditions (evangelical / catholic / mainline) are a first-class parameter, chosen by the user.
6. **Graceful degradation.** No Gloo keys: curated map + a soft note. All voice vendors down: pre-generated Twi audio still plays and the verse is still readable. ASR fails: the UI falls back to typing.

## The open-model horizon: DONDO

Dawuro already ships open models: every curated and daily verse is pre-synthesized in Twi with Meta's **MMS-tts-aka** and served as static audio, so voice works with no API. The next step is open-model **speech recognition**.

Days before this submission, GhanaNLP/Khaya AI open-sourced **DONDO** - 26 Apache-2.0 w2v-BERT 2.0 ASR checkpoints (21 monolingual + 5 multilingual, ~0.6B params each) covering 27 African language varieties ([arXiv:2607.21540](https://arxiv.org/abs/2607.21540), weights at [huggingface.co/KhayaAI](https://huggingface.co/KhayaAI)).

Two facts make this directly relevant:

1. **The published word-error rates are measured on religious read-speech - this app's exact domain.** Multilingual-checkpoint WERs for Dawuro's speech languages: Ewe 6.6%, Dagbani 12.3%, Kusaal 13.3%, Fante 13.4%, Twi 14.7%, Ga 16.0% (monolingual Ewe reaches 4.5%).
2. **Apache-2.0 licensing makes a self-hosted ASR fallback tier legally clean.** The hosted Khaya API is a single point of failure (we hit a live outage during development); a single Southern-Ghana multilingual checkpoint (Twi + Ewe + Fante + Ga in one model) on a small GPU or an HF Inference Endpoint would keep speak-your-heart input alive when the hosted API is down. CTC single-pass inference also sidesteps the long-audio timeout pattern.

The roadmap: hosted-first, open-model fallback for both TTS (already live) and ASR (DONDO), and eventually Scripture-audio fine-tuning under Apache-2.0.

## Verify it yourself

- Live app: https://dawuro.vercel.app
- Repository: https://github.com/NasamuAlhassan/dawuro

Three claims, each checkable in about 60 seconds:

1. **YouVersion is the only source of Scripture words.** Open `https://dawuro.vercel.app/api/health` - see `twiAccess` probing a real YouVersion passage. Then open `https://dawuro.vercel.app/v/tw/PHP.4.6-7`: the Twi text matches Asante Twi Nkwa Asem (ASNA) on YouVersion, with Biblica attribution. In the repo, `lib/gloo.ts` and `lib/verses.ts` contain no verse text anywhere - only USFM references.
2. **The receiver needs nothing.** Open `https://dawuro.vercel.app/v/tw/PHP.4.6-7` in a private window: verse in Twi + English, Play works, and the reply flow returns a verse - no login, no install. `https://dawuro.vercel.app/v/tw/PHP.4.6-7/opengraph-image` is the verse card WhatsApp shows in the chat.
3. **The voice is real and resilient.** On that page, tap Play: Twi plays from pre-generated MMS audio served as a static `/audio/tw/PHP.4.6-7.mp3` (no API - it works during vendor outages). Switch to English in Settings and Play: the MP3 comes from Microsoft Edge's neural voice (`X-Dawuro-Audio-Source: edge-tts` on `/api/speak`). When hosted Khaya TTS is up, it takes over automatically.